# Hybrid vs Classical diagnosis
A single pipeline notebook that diagnoses **whether the classifier head (classical or quantum) can be the source of any advantage** on the MicroscopyRGB task, or whether the bottleneck lies elsewhere (representation and rare-class information).

Every experiment reproduces the repository's **`freeze=full` regime** exactly: each trained head sits on the **same frozen ResNet34-MLL23 backbone** and is trained *through* it with the identical recipe used in `train.py`:
- Weak conditional augmentation
- BatchNorm adaptation
- Focal loss ($\gamma = 1$) with class-balanced weights
- Balanced sampler
- AdamW `lr_head=2e-2` + `ReduceLROnPlateau`
- 50 epochs

Metrics are **pooled out-of-fold** over 5 folds, averaged over 3 seeds, so the numbers are directly comparable to the classical campaign. 

The purely-sklearn diagnostics run on the **per-fold BN-adapted features** produced while Step 0 trains its linear head.

| Step | Question | Method |
|------|----------|--------|
| **0** | Is the frozen representation separable by a linear head? Is the head the bottleneck? | freeze=full **trained** linear head + k-NN probe |
| **1** | Does a more expressive head (incl. a quantum core) beat the linear head in the same regime? | heads trained through the backbone, param-matched cores |
| **2** | Do long-tail / few-shot *inference rules* lift rare-class recall / macro-F1? | logit-adjust, tau-norm, NCM on BN-adapted features |
| **3** | Does *adding information* (distribution calibration) raise rare-class macro-F1? | "Free Lunch" feature-space calibration (Yang et al. 2021) |

**Run the cells in order.**

Each step prints a summary table (balanced accuracy, macro-F1 with per-seed std, per-class recall) and writes its results to a JSON file under `results/`.

## 1. Setup

In [1]:
from __future__ import annotations

import json
import math
import os
import sys
from itertools import combinations
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import DataLoader

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import balanced_accuracy_score, f1_score, recall_score
from scipy import stats

In [2]:
# Define project root and scripts directory so data.* / models.* can be imported
_PROJECT_ROOT = Path("/home/quantum_secqdevops/HQCNNxLargeScaleMedicalImages")
_SCRIPTS_DIR = _PROJECT_ROOT / "scripts"
sys.path.insert(0, str(_SCRIPTS_DIR)) 

In [3]:
# Load MICROSCOPY_ROOT and MLL23_DATA_DIR from .env
for _line in (_PROJECT_ROOT / ".env").read_text().splitlines():
    _line = _line.strip()
    if _line and "=" in _line:
        _k, _, _v = _line.partition("=")
        os.environ.setdefault(_k.strip(), _v.strip())
        print(f"Set environment variable {_k.strip()}={_v.strip()}")

Set environment variable MICROSCOPY_ROOT=/mnt/dataset/microscopy
Set environment variable MLL23_DATA_DIR=/mnt/dataset/MML23


In [4]:
from data.dataset import get_dataset
from data.utils import make_stratified_splits
from models.classifiers import TensorNormalize
from models.hybrid import _ResNetFeatureExtractor

print("Imports OK")

Imports OK


## 2. Configuration

`CFG` is a **single config object**: dataset / cross-validation / backbone settings **plus** the exact `freeze=full` training recipe (loss, imbalance, augmentation, `lr_head`, …). 

Its attribute names match what `train.py`'s builders expect.

Key run-scope knobs:
- `CFG.epochs` (default 50),
- `CFG.seeds` (default `[9, 122, 250]`),
- `CFG.with_vqc = True` to add the quantum-core head in Step 1 (slow — trains a VQC through the backbone).

In [5]:

CFG = SimpleNamespace(
    dataset="MicroscopyRGB",
    data_dir=os.environ.get("MICROSCOPY_ROOT"),
    min_samples_per_class=15,
    exclude_classes=[],
    below_threshold_strategy="drop",
    missing_image_policy="drop",
    rare_class_name="RARE",
    backbone="resnet34",
    backbone_weights=(
        "results/MLL23/none/"
        "23-06-2026_18h18_mll23_final_backbone_resnet34_fullft_focal_g0p5_ls0p05_"
        "lr3em5_head1em3_weak_caug_lossimb/fold_1/backbone.pth"
    ),
    classifier_network="resnet34",
    classifier_pretrained=True,
    resnet_freeze_mode="full",
    processing_mode="none",
    loss_fn="focal", 
    focal_gamma=1, 
    label_smoothing=0,
    imbalance_handling="both", 
    sampler_mode="balanced", 
    undersample_strength=0.7,
    augmentation="weak", 
    conditional_augmentation=True, 
    rare_class_threshold=50, 
    non_rare_aug_prob=0.3,
    lr=1e-4, 
    lr_head=0.02, 
    weight_decay=1e-3,
    epochs=50, 
    batch_size=64, 
    num_workers=4, 
    freeze_processor=False,
    seeds=[9, 122, 250],
    n_splits=5, 
    cv_strategy="kfold",
    with_vqc=False,
    device="cuda",
    results_dir=str(_PROJECT_ROOT / "results"),
)
os.makedirs(CFG.results_dir, exist_ok=True)

print("Config ready (single object; freeze=full recipe):")
print(f"  - seeds x folds = {len(CFG.seeds)} x {CFG.n_splits}")
print(f"  - loss={CFG.loss_fn}\n     - g{CFG.focal_gamma}\n     - imb={CFG.imbalance_handling}"
      f"\n     - aug={CFG.augmentation}(cond)\n     - lr_head={CFG.lr_head}\n     - freeze={CFG.resnet_freeze_mode}")
print(f"  - backbone weights: MLL23")

Config ready (single object; freeze=full recipe):
  - seeds x folds = 3 x 5
  - loss=focal
     - g1
     - imb=both
     - aug=weak(cond)
     - lr_head=0.02
     - freeze=full
  - backbone weights: MLL23


In [6]:
def _ci95(values):
    """Mean, std, and 95% CI half-width over a list of measurements."""
    a = np.asarray(values, dtype=float)
    mean, std, n = a.mean(), a.std(ddof=1), len(a)
    half = stats.t.ppf(0.975, df=n - 1) * std / np.sqrt(n) if n > 1 else 0.0
    return mean, std, half

## 3. Shared infrastructure

This cell sets up the machinery to reproduce the repository's `freeze=full` training **inside every experiment**, reusing `train.py`'s own building blocks:

- `_FrozenBackboneHead`: the frozen MLL23 backbone + a swappable head.
- `run_ff_head(head_name, seeds)`: trains a head **through** the frozen backbone (augmentation + BN adaptation live), 50 epochs, best-epoch on val macro-F1, and returns **pooled-OOF** metrics per seed.
- `FOLD_FEATS`: per-fold BN-adapted features, cached while Step 0 trains its linear head, and consumed by the sklearn-only steps.

In [7]:

# Keep the project-root ./wandb directory from shadowing the real package when importing train.py
sys.path = [p for p in sys.path if os.path.abspath(p or ".") != str(_PROJECT_ROOT)]
if str(_SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(_SCRIPTS_DIR))

from collections import Counter
from data.dataset import Subset, get_dataset
from data.augmentation import get_augmentation, set_global_seed
from models.classifiers import EndToEndModel, TensorNormalize
from models.hybrid import _ResNetFeatureExtractor
from utils.metrics import compute_metrics
import train as TR

DEVICE = torch.device(CFG.device)
_BACKBONE_WEIGHTS = str(_PROJECT_ROOT / CFG.backbone_weights)

N_QUBITS = 12
VQC_LAYERS = 1
VQC_READOUT = "bloch"
CORE_WIDTH = 3 * 12

In [8]:
# Definition of the swappable heads and cores for the frozen-backbone experiments.
class LinearHead(nn.Module):
    def __init__(self, in_dim, n_classes):
        super().__init__()
        self.net = nn.Linear(in_dim, n_classes)

    def forward(self, x): 
        return self.net(x)


class MLPHead(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=64, p=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim), 
            nn.Linear(in_dim, hidden), 
            nn.GELU(),
            nn.Dropout(p), 
            nn.Linear(hidden, n_classes)
        )

    def forward(self, x): 
        return self.net(x)


class _IdentityCore(nn.Module):
    def forward(self, x): 
        return x


class _Linear36Core(nn.Module):
    def __init__(self, w=CORE_WIDTH): 
        super().__init__()
        self.net = nn.Linear(w, w)

    def forward(self, x): 
        return self.net(x)


class _MLP36Core(nn.Module):
    def __init__(self, w=CORE_WIDTH):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(w, w), 
            nn.GELU(), 
            nn.Linear(w, w)
        )
    
    def forward(self, x): 
        return self.net(x)

class AdapterHead(nn.Module):
    """Shared trunk + swappable 36->36 core (the controlled comparison)."""
    def __init__(self, in_dim, n_classes, core, hidden=64, p=0.2):
        super().__init__()
        
        self.trunk = nn.Sequential(
            nn.LayerNorm(in_dim), 
            nn.Linear(in_dim, hidden), 
            nn.GELU(),
            nn.Dropout(p), 
            nn.Linear(hidden, CORE_WIDTH)
        )
        
        self.angle_scale = nn.Parameter(torch.tensor(0.75 * math.pi))
        self.core = core
        self.bn = nn.BatchNorm1d(CORE_WIDTH)
        self.proj = nn.Linear(CORE_WIDTH, n_classes)
    
    def forward(self, x):
        z = torch.tanh(self.trunk(x)) * self.angle_scale
        z = self.core(z)
        if z.device != x.device: z = z.to(x.device)
        return self.proj(self.bn(z))

def build_head(name, in_dim, n_classes):
    if name == "linear": return LinearHead(in_dim, n_classes)
    if name == "mlp": return MLPHead(in_dim, n_classes)
    if name == "adapter_identity": return AdapterHead(in_dim, n_classes, _IdentityCore())
    if name == "adapter_linear36": return AdapterHead(in_dim, n_classes, _Linear36Core())
    if name == "adapter_mlp36": return AdapterHead(in_dim, n_classes, _MLP36Core())
    if name == "adapter_vqc":
        from models.hybrid import VariationalQuantumLayer
        return AdapterHead(in_dim, n_classes, VariationalQuantumLayer(N_QUBITS, VQC_LAYERS, readout=VQC_READOUT))
    raise ValueError(f"Unknown head '{name}'.")

def n_trainable(m): 
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

class _FrozenBackboneHead(nn.Module):
    """Frozen MLL23 backbone (freeze=full) + swappable head.
    For LinearHead this is architecturally identical to the repo's resnet fc."""
    def __init__(self, head):
        super().__init__()
        self.backbone = _ResNetFeatureExtractor(
            backbone=CFG.backbone, weights_path=_BACKBONE_WEIGHTS, pretrained=True, freeze_mode="full")
        self.head = head
    def forward(self, x):
        return self.head(self.backbone(x))

In [9]:
FOLD_FEATS = {}

@torch.no_grad()
def _extract_feats(model, dataset, idx, normalization):
    model.eval()
    dl = DataLoader(
        Subset(dataset, idx), 
        batch_size=CFG.batch_size, 
        shuffle=False, 
        num_workers=CFG.num_workers
    )
    
    feats, labs = [], []
    
    for imgs, lbls in dl:
        f = model.classifier.backbone(normalization(imgs.to(DEVICE)))
        
        feats.append(f.cpu().numpy())
        labs.append(np.asarray(lbls))
    
    return np.concatenate(feats).astype(np.float32), np.concatenate(labs).astype(np.int64)


def run_ff_head(head_name, seeds, cache_features=False):
    """
    Train `head_name` THROUGH the frozen backbone with the exact freeze=full
    recipe (CFG).
    
    Return per-seed pooled-OOF metrics + trainable-param count.
    
    If cache_features, also populate FOLD_FEATS from the BN-adapted backbone.
    """
    per_seed, n_params = {}, None
    
    # Loop over seeds
    for seed in seeds:
        set_global_seed(seed)

        splits = make_stratified_splits(SimpleNamespace(labels=dataset.labels), CFG.n_splits, seed, strategy=CFG.cv_strategy)
        
        oof_l, oof_p = [], []
        for fold, (train, val) in enumerate(splits):

            class_counts = Counter(dataset.labels[i] for i in train)
            train_labels = [dataset.labels[i] for i in train]

            _, normalization = TR._build_processor(CFG, dataset)   # ToTensor + TensorNormalize
            normalization = normalization.to(DEVICE)
            
            model = EndToEndModel(
                classifier=_FrozenBackboneHead(build_head(head_name, 512, n_classes)),
                processor=None, 
                normalization=normalization, 
                augmentation=get_augmentation(CFG.augmentation),
                augmentation_target_classes=sorted(i for i in range(n_classes) if class_counts.get(i, 0) <= CFG.rare_class_threshold),
                non_target_augmentation_prob=CFG.non_rare_aug_prob, freeze_processor=False,
            ).to(DEVICE)
            
            if n_params is None: 
                n_params = n_trainable(model)

            criterion = TR._build_criterion(
                CFG, 
                class_counts, 
                n_classes, 
                DEVICE
            )
            
            optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                          lr=CFG.lr_head, weight_decay=CFG.weight_decay)
            
            scheduler = TR._build_scheduler(CFG, optimizer)
            sampler, _ = TR._build_sampler(CFG, train_labels, class_counts)

            kw = dict(
                batch_size=CFG.batch_size, 
                num_workers=CFG.num_workers, 
                pin_memory=DEVICE.type == "cuda"
            )

            tl = DataLoader(
                Subset(dataset, train), 
                sampler=sampler, 
                shuffle=(sampler is None), 
                **kw
            )

            vl = DataLoader(
                Subset(dataset, val), 
                shuffle=False, 
                **kw
            )

            best_f1, best = -1.0, None
            for _ in range(CFG.epochs):
                TR.train_one_epoch(
                    model, 
                    tl, 
                    criterion, 
                    optimizer, DEVICE, 
                    display_names
                )

                _, vm, vlab, vpred = TR.validate(
                    model, 
                    vl, 
                    criterion, 
                    DEVICE, 
                    display_names
                )

                scheduler.step(vm["macro_f1"])

                if vm["macro_f1"] > best_f1:
                    best_f1, best = vm["macro_f1"], (vlab.copy(), vpred.copy())

            oof_l.append(best[0])
            oof_p.append(best[1])
            
            if cache_features:
                Xtr, ytr = _extract_feats(model, dataset, train, normalization)
                Xva, yva = _extract_feats(model, dataset, val, normalization)
                FOLD_FEATS[(seed, fold)] = (Xtr, ytr, Xva, yva)

            print(f"    [{head_name}] seed {seed} fold {fold + 1}/{CFG.n_splits} done")

        per_seed[seed] = compute_metrics(np.concatenate(oof_p), np.concatenate(oof_l), display_names)
    return per_seed, n_params

def pooled_over_feats(predict_fn, methods, seeds):
    """
    Run sklearn-style `predict_fn(Xtr,ytr,Xva,yva)->{method:preds}` over the
    cached per-fold BN-adapted features
    
    return {method: {seed: metrics}} pooled-OOF.
    """
    acc = {
        m: {
            s: {
                "l": [], 
                "p": []
            } for s in seeds
        } for m in methods
    }
    
    for (seed, fold), (Xtr, ytr, Xva, yva) in sorted(FOLD_FEATS.items()):
        if seed not in seeds:
            continue

        preds = predict_fn(Xtr, ytr, Xva, yva)
        for m in methods:
            acc[m][seed]["l"].append(yva)
            acc[m][seed]["p"].append(preds[m])
    
    out = {}
    for m in methods:
        out[m] = {
            s: compute_metrics(
                np.concatenate(acc[m][s]["p"]), 
                np.concatenate(acc[m][s]["l"]), 
                display_names
                )  for s in seeds
        }
    return out

def agg(per_seed):
    """(ba_mean, ba_std, mf1_mean, mf1_std, per_class_recall[array]) over seeds."""
    ms = list(per_seed.values())
    ba = np.array([m["balanced_accuracy"] for m in ms])
    mf = np.array([m["macro_f1"] for m in ms])
    pcr = np.array([[r["recall"] for r in m["per_class"]] for m in ms]).mean(0)
    return float(ba.mean()), float(ba.std()), float(mf.mean()), float(mf.std()), pcr


In [10]:
# Build the image dataset once
dataset = get_dataset(
    CFG.dataset, 
    root_dir=CFG.data_dir, 
    min_samples_per_class=CFG.min_samples_per_class,
    exclude_classes=CFG.exclude_classes, 
    below_threshold_strategy=CFG.below_threshold_strategy,
    rare_class_name=CFG.rare_class_name, 
    missing_image_policy=CFG.missing_image_policy,
)

class_names = list(getattr(dataset, "classes", sorted(set(dataset.labels))))
display_names = list(getattr(dataset, "display_classes", class_names))
n_classes = len(class_names)

didx = {display_names[i]: i for i in range(n_classes)}

total_counts = np.bincount(np.asarray(dataset.labels), minlength=n_classes)

show_rare = [r for r in ("BAS", "LYA", "EOS") if r in didx]

rare_names = [display_names[i] for i in range(n_classes) if int(total_counts[i]) < CFG.rare_class_threshold]

print(f"Regime: freeze=full through frozen MLL23 backbone")
print(f"    epochs={CFG.epochs}")
print(f"    seeds={CFG.seeds}", end="\n\n")

print(f"Dataset: {type(dataset).__name__} [{len(dataset)} samples, {n_classes} classes]")
print(f"{n_classes} classes: {display_names}")
print("Per-class totals:")
for i in range(n_classes):
    tag = " <- rare" if int(total_counts[i]) < CFG.rare_class_threshold else ""
    print(f"  {display_names[i]:<6} {int(total_counts[i])}{tag}")

Regime: freeze=full through frozen MLL23 backbone
    epochs=50
    seeds=[9, 122, 250]

Dataset: MicroscopyDatasetRGB [821 samples, 7 classes]
7 classes: ['BAS', 'EOS', 'KSC', 'LYA', 'LYT', 'MON', 'NGS']
Per-class totals:
  BAS    18 <- rare
  EOS    43 <- rare
  KSC    158
  LYA    30 <- rare
  LYT    224
  MON    180
  NGS    168


## 4. Frozen-backbone separability

The **linear head is trained through the frozen backbone** with the exact `freeze=full` recipe. 

It is the **baseline the richer heads (Step 1) must beat**: if a more expressive (non-linear or quantum) head does not improve on it *in this same regime*, the head is not the bottleneck and the limit lies in the representation (or the number of samples).

Training the linear head also **BN-adapts** the backbone: we cache the resulting per-fold features (`FOLD_FEATS`) for the sklearn steps. 

The k-NN probe is a pure feature diagnostic on those features.

In [11]:
print("Training the linear head through the frozen backbone (freeze=full recipe)...")

linear_per_seed, linear_params = run_ff_head("linear", CFG.seeds, cache_features=True)

def _knn_predict(Xtr, ytr, Xva, yva):
    sc = StandardScaler().fit(Xtr)
    knn = KNeighborsClassifier(n_neighbors=5).fit(sc.transform(Xtr), ytr)
    return {"knn": knn.predict(sc.transform(Xva))}

knn_per_seed = pooled_over_feats(_knn_predict, ["knn"], CFG.seeds)["knn"]

print("\n" + "=" * 72)
print(f"STEP 0 freeze=full regime (pooled OOF, {len(CFG.seeds)} seeds x {CFG.n_splits} folds)")
print(f"Random-baseline balanced accuracy = 1/{n_classes} = {1.0 / n_classes:.3f}")
print("=" * 72)

summary = {
    "n_classes": n_classes, 
    "n_seeds": len(CFG.seeds), 
    "probes": {}
}

for name, ps in (("linear", linear_per_seed), ("knn", knn_per_seed)):
    ba_m, ba_s, f1_m, f1_s, pcr = agg(ps)
    print(f"\n{name:>7}:")
    print(f"   balanced acc = {ba_m:.4f} +/- {ba_s:.4f} (std over seeds)")
    print(f"   macro F1     = {f1_m:.4f} +/- {f1_s:.4f}")
    summary["probes"][name] = {
        "balanced_acc_mean": ba_m, 
        "balanced_acc_std": ba_s,
        "macro_f1_mean": f1_m, 
        "macro_f1_std": f1_s,
        "per_seed": {
            str(s): {
                "balanced_acc": m["balanced_accuracy"], 
                "macro_f1": m["macro_f1"]
                } for s, m in ps.items()
        },
        "per_class_recall": {
            display_names[i]: float(pcr[i]) for i in range(n_classes)
        },
    }

    print("   per-class recall (mean):")
    for i in range(n_classes):
        tag = " <- rare" if int(total_counts[i]) < CFG.rare_class_threshold else ""
        print(f"      {display_names[i]:<6} {pcr[i]:.3f}{tag}")

summary["probes"]["linear"]["n_params"] = linear_params

Training the linear head through the frozen backbone (freeze=full recipe)...
[Backbone] Loaded custom weights (resnet34) from /home/quantum_secqdevops/HQCNNxLargeScaleMedicalImages/results/MLL23/none/23-06-2026_18h18_mll23_final_backbone_resnet34_fullft_focal_g0p5_ls0p05_lr3em5_head1em3_weak_caug_lossimb/fold_1/backbone.pth (216 tensors)
    [linear] seed 9 fold 1/5 done
[Backbone] Loaded custom weights (resnet34) from /home/quantum_secqdevops/HQCNNxLargeScaleMedicalImages/results/MLL23/none/23-06-2026_18h18_mll23_final_backbone_resnet34_fullft_focal_g0p5_ls0p05_lr3em5_head1em3_weak_caug_lossimb/fold_1/backbone.pth (216 tensors)
    [linear] seed 9 fold 2/5 done
[Backbone] Loaded custom weights (resnet34) from /home/quantum_secqdevops/HQCNNxLargeScaleMedicalImages/results/MLL23/none/23-06-2026_18h18_mll23_final_backbone_resnet34_fullft_focal_g0p5_ls0p05_lr3em5_head1em3_weak_caug_lossimb/fold_1/backbone.pth (216 tensors)
    [linear] seed 9 fold 3/5 done
[Backbone] Loaded custom weights

In [12]:
# Readout
lin_mF1 = summary["probes"]["linear"]["macro_f1_mean"]
lin_ba = summary["probes"]["linear"]["balanced_acc_mean"]
knn_mF1 = summary["probes"]["knn"]["macro_f1_mean"]
knn_ba = summary["probes"]["knn"]["balanced_acc_mean"]

print("=" * 72)
print("READOUT: Step 0 - freeze=full trained linear head vs kNN diagnostic")
print("=" * 72)

print(f"  linear head (through backbone)  BA = {lin_ba:.4f}   macro F1 = {lin_mF1:.4f}")
print(f"  kNN (BN-adapted features)       BA = {knn_ba:.4f}   macro F1 = {knn_mF1:.4f}")
print("\nThe trained linear head is the freeze=full baseline the richer heads (Step 1) must beat.")

out = os.path.join(CFG.results_dir, "linear_probe.json")
with open(out, "w") as f:
    json.dump(summary, f, indent=2, default=float)
print(f"\nSaved: {out}")

READOUT: Step 0 - freeze=full trained linear head vs kNN diagnostic
  linear head (through backbone)  BA = 0.8278   macro F1 = 0.7865
  kNN (BN-adapted features)       BA = 0.6247   macro F1 = 0.6379

The trained linear head is the freeze=full baseline the richer heads (Step 1) must beat.

Saved: /home/quantum_secqdevops/HQCNNxLargeScaleMedicalImages/results/linear_probe.json


## 5. Controlled head comparison

Every head is trained **through the frozen backbone** with the identical `freeze=full` recipe. 

Only the head architecture varies, so **any difference is attributable to the head.**

The `adapter_*` heads share one trunk and differ **only** in the 36->36 core:

```
backbone(512) -> LayerNorm -> Linear(512,64) -> GELU -> Dropout -> Linear(64, 36) -> tanh*angle_scale -> [ CORE: 36->36 ] -> BatchNorm(36) -> Linear(36, C)
```

`CORE in {identity, linear36, mlp36, vqc}`.

Set `CFG.with_vqc = True` (in the config cell) to add the quantum core.

In [13]:
# Step 1 trains the SAME heads THROUGH the frozen backbone with the freeze=full recipe. 

# Only the head architecture varies

#`linear` is reused from Step 0 (same head, same recipe).

heads = ["linear", "mlp", "adapter_identity", "adapter_linear36", "adapter_mlp36"]
if CFG.with_vqc:
    heads = heads + ["adapter_vqc"]

print(f"Heads: {heads}")
print(f"Regime: freeze=full through frozen backbone "
      f"{len(CFG.seeds)} seeds x {CFG.n_splits} folds x {CFG.epochs} epochs")

Heads: ['linear', 'mlp', 'adapter_identity', 'adapter_linear36', 'adapter_mlp36']
Regime: freeze=full through frozen backbone 3 seeds x 5 folds x 50 epochs


In [14]:
# Train each head through the frozen backbone (pooled-OOF per seed).
# `linear` is reused from Step 0 to avoid retraining the identical head.
records = {}
for h in heads:
    if h == "linear":
        records[h] = {
            "per_seed": linear_per_seed, 
            "n_params": linear_params
        }
        print(f"=== head: linear (reused from Step 0) ===")
        continue
    
    print(f"=== head: {h} ===")
    per_seed, n_params = run_ff_head(h, CFG.seeds)
    
    records[h] = {
        "per_seed": per_seed, 
        "n_params": n_params
    }

=== head: linear (reused from Step 0) ===
=== head: mlp ===
[Backbone] Loaded custom weights (resnet34) from /home/quantum_secqdevops/HQCNNxLargeScaleMedicalImages/results/MLL23/none/23-06-2026_18h18_mll23_final_backbone_resnet34_fullft_focal_g0p5_ls0p05_lr3em5_head1em3_weak_caug_lossimb/fold_1/backbone.pth (216 tensors)
    [mlp] seed 9 fold 1/5 done
[Backbone] Loaded custom weights (resnet34) from /home/quantum_secqdevops/HQCNNxLargeScaleMedicalImages/results/MLL23/none/23-06-2026_18h18_mll23_final_backbone_resnet34_fullft_focal_g0p5_ls0p05_lr3em5_head1em3_weak_caug_lossimb/fold_1/backbone.pth (216 tensors)
    [mlp] seed 9 fold 2/5 done
[Backbone] Loaded custom weights (resnet34) from /home/quantum_secqdevops/HQCNNxLargeScaleMedicalImages/results/MLL23/none/23-06-2026_18h18_mll23_final_backbone_resnet34_fullft_focal_g0p5_ls0p05_lr3em5_head1em3_weak_caug_lossimb/fold_1/backbone.pth (216 tensors)
    [mlp] seed 9 fold 3/5 done
[Backbone] Loaded custom weights (resnet34) from /home/qua

In [16]:
# Summary
print("=" * 72)
print(f"STEP 1 - heads trained THROUGH frozen backbone (freeze=full), pooled OOF, "
      f"{len(CFG.seeds)} seeds")
print(f"reference: Step 0 linear head  BA {lin_ba:.3f} / mF1 {lin_mF1:.3f}")
print("=" * 72)

hdr = f"{'head':<18}{'#params':>9}  {'balanced_acc':<22}{'macro_f1':<22}"
hdr += "".join(f"{r:>6}" for r in show_rare)
print(hdr)

summary = {
    "n_classes": n_classes, 
    "regime": "freeze_full_through_backbone", 
    "heads": {}
}

for h in heads:
    ba_m, ba_s, f1_m, f1_s, pcr = agg(records[h]["per_seed"])
    rr = "".join(f"{(pcr[didx[r]] if r in didx else float('nan')):>6.2f}" for r in show_rare)
    print(f"{h:<18}{records[h]['n_params']:>9,}  {ba_m:.4f} +/- {ba_s:.4f}      "
          f"{f1_m:.4f} +/- {f1_s:.4f}   {rr}")
    
    summary["heads"][h] = {
        "n_params": records[h]["n_params"],
        "balanced_acc_mean": ba_m, 
        "balanced_acc_std": ba_s,
        "macro_f1_mean": f1_m, 
        "macro_f1_std": f1_s,
        "per_seed": {
            str(s): {
                "balanced_acc": m["balanced_accuracy"], 
                "macro_f1": m["macro_f1"]
            } for s, m in records[h]["per_seed"].items()
        },
        "per_class_recall": {
            display_names[i]: float(pcr[i]) for i in range(n_classes)
        },
    }

print("\nReading guide (all heads in the SAME freeze=full regime):")
print(f"  Richer heads macro_f1 >> linear head ({lin_mF1:.3f})?")
print(f"        Then head expressivity adds over the linear head in the deployed regime.")
print(f"  All heads ~{lin_mF1:.3f}?")
print(f"        Then the head is not the bottleneck even here -> representation + rare-sample limit.")
if "adapter_vqc" in heads:
    print("  adapter_vqc vs adapter_identity: does the quantum core add over just the adapter?")
    print("  adapter_vqc vs adapter_linear36/mlp36 (matched capacity): quantum vs classical core.")
else:
    print("  (set CFG.with_vqc = True in the infra cell to add the quantum-core comparison)")

out = os.path.join(CFG.results_dir, "head_comparison.json")
with open(out, "w") as f:
    json.dump(summary, f, indent=2, default=float)
print(f"\nSaved: {out}")

STEP 1 - heads trained THROUGH frozen backbone (freeze=full), pooled OOF, 3 seeds
reference: Step 0 linear head  BA 0.828 / mF1 0.786
head                #params  balanced_acc          macro_f1                 BAS   LYA   EOS
linear                3,591  0.8278 +/- 0.0028      0.7865 +/- 0.0009     0.72  0.64  0.88
mlp                  34,311  0.8048 +/- 0.0149      0.7584 +/- 0.0051     0.65  0.64  0.86
adapter_identity     36,528  0.7696 +/- 0.0039      0.7054 +/- 0.0144     0.65  0.69  0.87
adapter_linear36     37,860  0.7636 +/- 0.0124      0.7020 +/- 0.0089     0.69  0.67  0.88
adapter_mlp36        39,192  0.7708 +/- 0.0125      0.6851 +/- 0.0151     0.80  0.72  0.89

Reading guide (all heads in the SAME freeze=full regime):
  Richer heads macro_f1 >> linear head (0.786)?
        Then head expressivity adds over the linear head in the deployed regime.
  All heads ~0.786?
        Then the head is not the bottleneck even here -> representation + rare-sample limit.
  (set CFG.with_vq

## 6. Long-tail and few-shot inference rules

We showed that head *capacity* does not help. Do cheap long-tail / few-shot **inference rules** on the freeze=full representation (the per-fold **BN-adapted features** cached in Step 0) lift rare-class recall (and thus macro-F1) over a plain linear classifier?

- `linear_plain`: multinomial LR, no class weighting (ERM)
- `linear_weighted`: LR with `class_weight='balanced'` (running reference)
- `logit_adjusted`: $\text{logits} − \tau \cdot \log(\text{prior})$ (Menon et al. 2021)
- `tau_norm`: weights L2-normalised by $\| w_c \|^\tau$, no bias (Kang et al. 2020)
- `ncm_cosine`: nearest class-mean prototype, cosine (Snell et al. 2017)

In [17]:
# Long-tail / few-shot inference rules, now applied to the freeze=full
# representation: the per-fold BN-adapted features cached in Step 0 (FOLD_FEATS).
LONGTAIL_PARAMS = SimpleNamespace(tau_la=1.0, tau_norm=1.0)


def _l2(x, eps=1e-8):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)


def _longtail_predict(Xtr, ytr, Xva, yva):
    sc = StandardScaler().fit(Xtr)
    Xtr, Xva = sc.transform(Xtr), sc.transform(Xva)
    preds = {}
    base = LogisticRegression(C=1.0, max_iter=5000).fit(Xtr, ytr)
    W, b = base.coef_, base.intercept_
    logits = Xva @ W.T + b
    preds["linear_plain"] = logits.argmax(1)

    counts = np.bincount(ytr, minlength=n_classes).astype(float)
    log_prior = np.log(np.clip(counts / counts.sum(), 1e-12, None))
    preds["logit_adjusted"] = (logits - LONGTAIL_PARAMS.tau_la * log_prior).argmax(1)

    wnorm = np.linalg.norm(W, axis=1, keepdims=True)
    W_tau = W / (wnorm ** LONGTAIL_PARAMS.tau_norm + 1e-12)
    preds["tau_norm"] = (Xva @ W_tau.T).argmax(1)

    wclf = LogisticRegression(C=1.0, class_weight="balanced", max_iter=5000).fit(Xtr, ytr)
    preds["linear_weighted"] = wclf.predict(Xva)

    Xtr_n, Xva_n = _l2(Xtr), _l2(Xva)
    protos = _l2(np.stack([Xtr_n[ytr == c].mean(0) for c in range(n_classes)]))
    preds["ncm_cosine"] = (Xva_n @ protos.T).argmax(1)
    return preds


methods = ["linear_plain", "linear_weighted", "logit_adjusted", "tau_norm", "ncm_cosine"]
res = pooled_over_feats(_longtail_predict, methods, CFG.seeds)

print("=" * 84)
print(f"Long-tail / few-shot rules on the freeze=full representation (BN-adapted "
      f"features)  ({len(CFG.seeds)} seeds x {CFG.n_splits} folds, pooled OOF)")
print(f"rare-class recall shown for {show_rare}")
print("=" * 84)
print(f"{'method':<17}{'balanced_acc':<22}{'macro_f1':<22}" + "".join(f"{r:>7}" for r in show_rare))

summary = {"regime": "freeze_full_bn_adapted_features", "methods": {}}
for m in methods:
    ba_m, ba_s, f1_m, f1_s, pcr = agg(res[m])
    print(f"{m:<17}{ba_m:.4f} +/- {ba_s:.4f}     {f1_m:.4f} +/- {f1_s:.4f}     "
          + "".join(f"{pcr[didx[r]]:>7.2f}" for r in show_rare))
    summary["methods"][m] = {
        "balanced_acc_mean": ba_m, "balanced_acc_std": ba_s,
        "macro_f1_mean": f1_m, "macro_f1_std": f1_s,
        "per_class_recall": {display_names[i]: float(pcr[i]) for i in range(n_classes)},
    }

base_f1 = summary["methods"]["linear_weighted"]["macro_f1_mean"]
best = max(methods, key=lambda m: summary["methods"][m]["macro_f1_mean"])
print(f"\nReference: linear_weighted macro_f1 = {base_f1:.4f}")
print(f"Best method: {best} (macro_f1 = {summary['methods'][best]['macro_f1_mean']:.4f}, "
      f"delta = {summary['methods'][best]['macro_f1_mean'] - base_f1:+.4f})")
print("Reading: if all tie the linear_weighted ref, the rare-class limit is information (samples), not the rule.")

out = os.path.join(CFG.results_dir, "fewshot_longtail.json")
with open(out, "w") as f:
    json.dump(summary, f, indent=2, default=float)
print(f"Saved: {out}")

Long-tail / few-shot rules on the freeze=full representation (BN-adapted features)  (3 seeds x 5 folds, pooled OOF)
rare-class recall shown for ['BAS', 'LYA', 'EOS']
method           balanced_acc          macro_f1                  BAS    LYA    EOS
linear_plain     0.7176 +/- 0.0103     0.7359 +/- 0.0086        0.30   0.24   0.78
linear_weighted  0.7494 +/- 0.0103     0.7562 +/- 0.0102        0.39   0.36   0.84
logit_adjusted   0.7814 +/- 0.0161     0.7600 +/- 0.0086        0.56   0.48   0.85
tau_norm         0.7823 +/- 0.0026     0.7289 +/- 0.0008        0.57   0.54   0.91
ncm_cosine       0.7027 +/- 0.0059     0.6450 +/- 0.0074        0.67   0.42   0.78

Reference: linear_weighted macro_f1 = 0.7562
Best method: logit_adjusted (macro_f1 = 0.7600, delta = +0.0037)
Reading: if all tie the linear_weighted ref, the rare-class limit is information (samples), not the rule.
Saved: /home/quantum_secqdevops/HQCNNxLargeScaleMedicalImages/results/fewshot_longtail.json


## 7. Feature-space distribution calibration

Re-weighting rules raise rare-class *recall* and balanced accuracy but not macro-F1 (they trade recall for precision).

This step tries the one few-shot method that *adds information*: distribution calibration (Yang, Liu & Xu, ICLR 2021), applied to the freeze=full representation (the per-fold **BN-adapted features**). It transfers feature-space variance from many-shot (base) classes to the rare classes and samples synthetic rare-class features.

Methods:
- `linear_weighted` (ref),
- `tukey_weighted` (Tukey only),
- `dist_calib` (the method),
- `dist_calib_la` (dist_calib + logit adjustment).

The key number is macro-F1 vs the reference. Only this method can raise it *if* rare-class information is recoverable by borrowing base-class variance.

In [18]:
# Distribution calibration (Yang et al. 2021) on the freeze=full representation:
# the per-fold BN-adapted features (FOLD_FEATS). Pooled OOF over seeds/folds.
FEAT_PARAMS = SimpleNamespace(rare_thr=60, lam=0.5, k=2, num_sampled=30, alpha_frac=0.1)


def tukey(x, lam):
    return np.power(x, lam) if lam != 0 else np.log(np.clip(x, 1e-8, None))


def calibrate_and_sample(Xtr_t, ytr, base_ids, rare_ids, k, num_sampled, alpha_frac, rng):
    base_means = {b: Xtr_t[ytr == b].mean(0) for b in base_ids}
    base_cov = {b: np.cov(Xtr_t[ytr == b], rowvar=False) for b in base_ids}
    base_M = np.stack([base_means[b] for b in base_ids])
    ridge = alpha_frac * np.mean([np.mean(np.diag(base_cov[b])) for b in base_ids])

    chol = {}
    for combo in combinations(range(len(base_ids)), k):
        Sigma = np.mean([base_cov[base_ids[i]] for i in combo], axis=0) + ridge * np.eye(base_M.shape[1])
        chol[combo] = np.linalg.cholesky(Sigma)

    synth_X, synth_y = [], []
    for c in rare_ids:
        for x in Xtr_t[ytr == c]:
            combo = tuple(sorted(np.argsort(((base_M - x) ** 2).sum(1))[:k]))
            mu = (sum(base_M[i] for i in combo) + x) / (k + 1)
            z = rng.standard_normal((num_sampled, x.shape[0]))
            synth_X.append(mu + z @ chol[combo].T)
            synth_y.append(np.full(num_sampled, c))
    if not synth_X:
        return np.empty((0, Xtr_t.shape[1])), np.empty((0,), dtype=int)
    return np.concatenate(synth_X), np.concatenate(synth_y)


methods = ["linear_weighted", "tukey_weighted", "dist_calib", "dist_calib_la"]
acc = {m: {s: {"l": [], "p": []} for s in CFG.seeds} for m in methods}

for seed in CFG.seeds:
    rng = np.random.default_rng(seed)
    for fold in range(CFG.n_splits):
        if (seed, fold) not in FOLD_FEATS:
            continue
        Xtr0, ytr, Xva0, yva = FOLD_FEATS[(seed, fold)]
        Xtr0 = np.clip(Xtr0.astype(np.float64), 0, None)   # backbone feats are post-ReLU >= 0
        Xva0 = np.clip(Xva0.astype(np.float64), 0, None)
        counts = np.bincount(ytr, minlength=n_classes)
        base_ids = [c for c in range(n_classes) if counts[c] >= FEAT_PARAMS.rare_thr]
        rare_ids = [c for c in range(n_classes) if 0 < counts[c] < FEAT_PARAMS.rare_thr]

        sc = StandardScaler().fit(Xtr0)
        clf = LogisticRegression(C=1.0, class_weight="balanced", max_iter=5000).fit(sc.transform(Xtr0), ytr)
        acc["linear_weighted"][seed]["l"].append(yva)
        acc["linear_weighted"][seed]["p"].append(clf.predict(sc.transform(Xva0)))

        Xtr_t, Xva_t = tukey(Xtr0, FEAT_PARAMS.lam), tukey(Xva0, FEAT_PARAMS.lam)
        sc_t = StandardScaler().fit(Xtr_t)
        clf_t = LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000).fit(sc_t.transform(Xtr_t), ytr)
        acc["tukey_weighted"][seed]["l"].append(yva)
        acc["tukey_weighted"][seed]["p"].append(clf_t.predict(sc_t.transform(Xva_t)))

        sx, sy = calibrate_and_sample(Xtr_t, ytr, base_ids, rare_ids,
                                      FEAT_PARAMS.k, FEAT_PARAMS.num_sampled, FEAT_PARAMS.alpha_frac, rng)
        Xaug, yaug = np.concatenate([Xtr_t, sx]), np.concatenate([ytr, sy])
        sc_a = StandardScaler().fit(Xaug)
        clf_dc = LogisticRegression(C=1.0, max_iter=2000).fit(sc_a.transform(Xaug), yaug)
        logits = sc_a.transform(Xva_t) @ clf_dc.coef_.T + clf_dc.intercept_
        acc["dist_calib"][seed]["l"].append(yva)
        acc["dist_calib"][seed]["p"].append(logits.argmax(1))

        prior = np.bincount(yaug, minlength=n_classes).astype(float)
        prior /= prior.sum()
        acc["dist_calib_la"][seed]["l"].append(yva)
        acc["dist_calib_la"][seed]["p"].append((logits - np.log(np.clip(prior, 1e-12, None))).argmax(1))

res = {m: {s: compute_metrics(np.concatenate(acc[m][s]["p"]), np.concatenate(acc[m][s]["l"]), display_names)
           for s in CFG.seeds} for m in methods}

print("=" * 84)
print(f"Distribution calibration on the freeze=full representation ({len(CFG.seeds)} seeds x {CFG.n_splits} folds, pooled OOF)")
print(f"  lambda={FEAT_PARAMS.lam} k={FEAT_PARAMS.k} num_sampled={FEAT_PARAMS.num_sampled} "
      f"alpha_frac={FEAT_PARAMS.alpha_frac} rare_thr={FEAT_PARAMS.rare_thr}")
print("=" * 84)
print(f"{'method':<17}{'balanced_acc':<22}{'macro_f1':<22}" + "".join(f"{r:>7}" for r in show_rare))

summary = {"regime": "freeze_full_bn_adapted_features", "methods": {}}
for m in methods:
    ba_m, ba_s, f1_m, f1_s, pcr = agg(res[m])
    print(f"{m:<17}{ba_m:.4f} +/- {ba_s:.4f}     {f1_m:.4f} +/- {f1_s:.4f}     "
          + "".join(f"{pcr[didx[r]]:>7.2f}" for r in show_rare))
    summary["methods"][m] = {
        "balanced_acc_mean": ba_m, "balanced_acc_std": ba_s,
        "macro_f1_mean": f1_m, "macro_f1_std": f1_s,
        "per_class_recall": {display_names[i]: float(pcr[i]) for i in range(n_classes)},
    }

ref = summary["methods"]["linear_weighted"]["macro_f1_mean"]
best = max(methods, key=lambda m: summary["methods"][m]["macro_f1_mean"])
print(f"\nReference linear_weighted macro_f1 = {ref:.4f}")
print(f"Best: {best} macro_f1 = {summary['methods'][best]['macro_f1_mean']:.4f} "
      f"(delta {summary['methods'][best]['macro_f1_mean'] - ref:+.4f})")
print("Reading: if dist_calib raises macro-F1 (not just recall), borrowing base variance recovered\n"
      "rare-class information. If it ties the ref, the rare-class limit is genuinely the samples.")

out = os.path.join(CFG.results_dir, "distribution_calibration.json")
with open(out, "w") as f:
    json.dump(summary, f, indent=2, default=float)
print(f"Saved: {out}")

Distribution calibration on the freeze=full representation (3 seeds x 5 folds, pooled OOF)
  lambda=0.5 k=2 num_sampled=30 alpha_frac=0.1 rare_thr=60
method           balanced_acc          macro_f1                  BAS    LYA    EOS
linear_weighted  0.7494 +/- 0.0103     0.7562 +/- 0.0102        0.39   0.36   0.84
tukey_weighted   0.7634 +/- 0.0020     0.7707 +/- 0.0040        0.43   0.39   0.85
dist_calib       0.7229 +/- 0.0129     0.6406 +/- 0.0086        0.59   0.56   0.95
dist_calib_la    0.7447 +/- 0.0097     0.6865 +/- 0.0038        0.57   0.43   0.95

Reference linear_weighted macro_f1 = 0.7562
Best: tukey_weighted macro_f1 = 0.7707 (delta +0.0144)
Reading: if dist_calib raises macro-F1 (not just recall), borrowing base variance recovered
rare-class information. If it ties the ref, the rare-class limit is genuinely the samples.
Saved: /home/quantum_secqdevops/HQCNNxLargeScaleMedicalImages/results/distribution_calibration.json
